# 02.2 — Ski resort scraping and price harmonisation

This notebook rebuilds the resort dataset from the current English-language Skiresort country listings. It targets **70 real resorts per country (280 total)**. It does not add placeholder resorts, overwrite rows with broad name-based rules, or impute missing prices. If a country cannot supply 70 usable source records, the notebook stops and reports the shortfall instead of manufacturing observations.

The English portal moved from `skiresort.info` to `skiresort.com` in August 2026. The listing pages expose resort names, elevation ranges, slope kilometres, lift counts and (where published) day-ticket prices. See the source links in the notebook and preserve the raw scrape for auditing.

In [ ]:
import os
import re
import time
import unicodedata
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.skiresort.com"
RAW_DIR = "../data/raw/skiresort_listing_pages"
PROCESSED_DIR = "../data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

HEADERS = {
    "User-Agent": "ZHAW-AlpineSkiResearch/1.0 (academic project; contact via repository owner)",
    "Accept-Language": "en-US,en;q=0.9",
}
REQUEST_DELAY_SECONDS = 1.5
TARGET_PER_COUNTRY = 70
COUNTRY_PAGES = {
    "CH": "switzerland",
    "IT": "italy",
    "AT": "austria",
    "FR": "france",
}

session = requests.Session()
session.headers.update(HEADERS)


def normalise_text(value):
    value = str(value).replace("\u200b", "").replace("\u200c", "").replace("\u200d", "")
    value = value.replace("\u2060", "").replace("\ufeff", "").replace("’", "'")
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", value).strip()


def to_number(value):
    if value is None:
        return np.nan
    return float(str(value).replace("'", "").replace(",", ".").strip())


def listing_card_text(anchor):
    """Find the smallest ancestor containing the resort's summary metrics."""
    for parent in [anchor, *anchor.parents]:
        text = " ".join(parent.stripped_strings)
        has_lifts = re.search(r"\b\d+\s+ski lifts?\b", text, re.I)
        has_elevation = re.search(r"\(\s*\d[\d,.]*\s*m\s*[-–—]\s*\d[\d,.]*\s*m\s*\)", text, re.I)
        if has_lifts and has_elevation and len(text) < 1800:
            return text
    return " ".join(anchor.parent.stripped_strings)


def parse_listing_card(name, country, url, card_text):
    text = normalise_text(card_text)
    elevation = re.search(r"\(\s*([\d,.]+)\s*m\s*[-–—]\s*([\d,.]+)\s*m\s*\)", text, re.I)
    slope = re.search(r"\)\s*([\d,.]+)\s*km\b", text, re.I)
    lifts = re.search(r"\b(\d+)\s+ski lifts?\b", text, re.I)

    base_m = top_m = slope_km = lifts_count = np.nan
    if elevation:
        base_m, top_m = to_number(elevation.group(1)), to_number(elevation.group(2))
        if not (300 <= base_m < top_m <= 5000):
            base_m = top_m = np.nan
    if slope:
        slope_km = to_number(slope.group(1))
        if not (0 < slope_km <= 650):
            slope_km = np.nan
    if lifts:
        lifts_count = int(lifts.group(1))
        if not (1 <= lifts_count <= 250):
            lifts_count = np.nan

    price_raw = price_eur = np.nan
    currency = None
    price_method = None
    chf = re.search(r"\b(?:SFr\.?|CHF)\s*([\d,.]+)", text, re.I)
    approx_eur = re.search(r"approx\.?\s*€\s*([\d,.]+)", text, re.I)
    direct_eur = re.search(r"€\s*([\d,.]+)", text)
    if chf:
        price_raw, currency = to_number(chf.group(1)), "CHF"
        if approx_eur:
            price_eur = to_number(approx_eur.group(1))
            price_method = "source_approx_eur"
        else:
            price_method = "chf_without_source_eur"
    elif direct_eur:
        price_raw = price_eur = to_number(direct_eur.group(1))
        currency, price_method = "EUR", "source_eur"

    return {
        "resort_name": name,
        "country": country,
        "url": url,
        "elevation_base_m": base_m,
        "elevation_top_m": top_m,
        "slope_km": slope_km,
        "lifts_count": lifts_count,
        "price_day_raw": price_raw,
        "currency": currency,
        "price_day_eur": price_eur,
        "price_method": price_method,
        "source_page_text": text,
    }

## 1. Discover resorts and extract listing data

The country pages are paginated. Each page is cached locally before parsing. The parser reads the resort summary card rather than relying on a stale local HTML folder. Cross-border areas are assigned once to a documented canonical country to avoid counting the same URL twice.

In [ ]:
CANONICAL_COUNTRY_BY_SLUG = {
    "les-portes-du-soleil": "FR",
    "via-lattea": "IT",
    "zermatt-breuil-cervinia-valtournenche-matterhorn": "CH",
    "ischgl-samnaun": "AT",
    "arosa-lenzerheide": "CH",
}

candidates = []
seen_urls = set()

for country, country_slug in COUNTRY_PAGES.items():
    country_count = 0
    for page_number in range(1, 4):
        suffix = "" if page_number == 1 else f"page/{page_number}/"
        page_url = f"{BASE_URL}/en/ski-resorts/{country_slug}/{suffix}"
        cache_path = os.path.join(RAW_DIR, f"{country}_{page_number}.html")
        try:
            response = session.get(page_url, timeout=30)
            response.raise_for_status()
            html = response.text
            with open(cache_path, "w", encoding="utf-8") as file:
                file.write(html)
        except requests.RequestException as exc:
            print(f"[WARN] {country} page {page_number} could not be fetched: {exc}")
            break

        soup = BeautifulSoup(html, "html.parser")
        page_added = 0
        for anchor in soup.find_all("a", href=True):
            href = urljoin(BASE_URL, anchor["href"].split("?")[0])
            parsed_url = urlparse(href)
            if parsed_url.netloc not in {"www.skiresort.com", "skiresort.com"}:
                continue
            if "/en/ski-resort/" not in parsed_url.path:
                continue
            canonical_url = href.rstrip("/") + "/"
            if canonical_url in seen_urls:
                continue
            name = normalise_text(anchor.get_text(" ", strip=True))
            if len(name) < 3:
                continue

            slug = parsed_url.path.rstrip("/").split("/")[-1]
            assigned_country = CANONICAL_COUNTRY_BY_SLUG.get(slug, country)
            card_text = listing_card_text(anchor)
            record = parse_listing_card(name, assigned_country, canonical_url, card_text)
            record["listing_country"] = country
            record["source_page"] = page_url
            record["source_retrieved_at_utc"] = pd.Timestamp.utcnow().isoformat()
            candidates.append(record)
            seen_urls.add(canonical_url)
            country_count += 1
            page_added += 1
            if country_count >= 100:
                break

        print(f"{country} page {page_number}: {page_added} new resort links; {country_count} collected so far")
        time.sleep(REQUEST_DELAY_SECONDS)
        if country_count >= 100:
            break

raw = pd.DataFrame(candidates)
if raw.empty:
    raise RuntimeError("No resort records were collected. Check network access and the Skiresort page structure.")

raw.to_csv(os.path.join(PROCESSED_DIR, "resorts_scrape_candidates.csv"), index=False)
print("\nCandidate records:", len(raw))
print(raw.groupby("country").size().rename("candidate_count"))

## 2. Quality checks, balanced sample and price harmonisation

The final sample is exactly 70 resorts per country (280 total). A resort must have a valid source URL and at least two observed structural/price fields among elevation range, slope length, lift count and ticket price. Missing values remain missing; no national-median substitution is performed. For Swiss resorts, the source's explicitly displayed approximate EUR price is used when available, while the original CHF price is retained separately.

In [ ]:
# Count observed fields and retain only records with evidence from the source card.
required_numeric = ["elevation_base_m", "elevation_top_m", "slope_km", "lifts_count", "price_day_raw"]
raw["observed_field_count"] = raw[required_numeric].notna().sum(axis=1)
raw["url_valid"] = raw["url"].str.startswith("https://www.skiresort.com/en/ski-resort/")

eligible = raw[raw["url_valid"] & (raw["observed_field_count"] >= 2)].copy()
# One canonical URL may occur on more than one country's listing page.
eligible = eligible.drop_duplicates(subset="url", keep="first")

country_counts = eligible.groupby("country").size().reindex(COUNTRY_PAGES, fill_value=0)
print("Eligible records by canonical country:")
print(country_counts)

short = country_counts[country_counts < TARGET_PER_COUNTRY]
if not short.empty:
    raise RuntimeError(
        "The source did not provide 70 eligible resorts for every country. "
        f"Available counts: {country_counts.to_dict()}. No synthetic rows were added. "
        "Inspect resorts_scrape_candidates.csv and adjust the listing pagination/parser before exporting."
    )

# Prefer cards with the greatest number of observed fields; stable order breaks ties.
selected = (
    eligible.sort_values(["country", "observed_field_count", "resort_name"],
                         ascending=[True, False, True], kind="mergesort")
            .groupby("country", group_keys=False)
            .head(TARGET_PER_COUNTRY)
            .copy()
)
selected["price_imputed"] = False
selected["manually_added"] = False
selected["scrape_source"] = "Skiresort.com country listing"

# Keep source-level price fields and never invent an exchange rate.
selected["price_day_eur"] = pd.to_numeric(selected["price_day_eur"], errors="coerce")
selected["price_day_raw"] = pd.to_numeric(selected["price_day_raw"], errors="coerce")

output_columns = [
    "resort_name", "country", "url", "elevation_base_m", "elevation_top_m",
    "slope_km", "lifts_count", "price_day_raw", "currency", "price_day_eur",
    "price_method", "price_imputed", "manually_added", "observed_field_count",
    "scrape_source", "source_page", "source_retrieved_at_utc"
]
clean = selected[output_columns].sort_values(["country", "resort_name"]).reset_index(drop=True)

assert len(clean) == 280, f"Expected 280 records, found {len(clean)}"
assert clean.groupby("country").size().eq(70).all(), "Country allocation is not balanced"
assert clean["url"].is_unique, "Duplicate source URLs remain"

output_path = os.path.join(PROCESSED_DIR, "resorts_clean.csv")
clean.to_csv(output_path, index=False)

print("\nFINAL DATASET EXPORTED:", output_path)
print("Total resorts:", len(clean))
print("Country allocation:")
print(clean.groupby("country").size())
print("Missing EUR ticket prices:", clean["price_day_eur"].isna().sum())
print("Missing slope lengths:", clean["slope_km"].isna().sum())
print("Manually added records:", int(clean["manually_added"].sum()))
display(clean.head(10))

## 3. Final validation

The checks below are intentionally strict: they confirm the 280-row target, the country balance, unique source URLs and the absence of manually fabricated records. They also make missing prices visible rather than hiding them through imputation.

In [ ]:
summary = clean.groupby("country").agg(
    resorts=("resort_name", "count"),
    avg_slope_km=("slope_km", "mean"),
    avg_price_eur=("price_day_eur", "mean"),
    missing_prices=("price_day_eur", lambda s: int(s.isna().sum())),
    missing_elevation=("elevation_base_m", lambda s: int(s.isna().sum())),
)
print(summary.round(2))
print("\nDuplicate URLs:", int(clean["url"].duplicated().sum()))
print("Manual records:", int(clean["manually_added"].sum()))
print("Price imputation flags:", int(clean["price_imputed"].sum()))
assert summary["resorts"].eq(70).all()
assert not clean["manually_added"].any()
assert not clean["price_imputed"].any()